# SQL Analytics Layer

## Objective

This notebook creates a structured SQL analytics layer for the AI Financial Intelligence Analyst project.

The analytical outputs from previous notebooks will be loaded into a SQLite database so that complaint intelligence can be queried using standard SQL.

The database will support:

- product-level KPI analysis;
- company and company–product comparisons;
- issue-severity analysis;
- sentiment and topic exploration;
- monthly complaint trends;
- company ranking queries;
- dashboard-ready analytical views.

## Why SQLite?

SQLite was selected because it:

- requires no database server;
- uses standard SQL;
- creates one portable database file;
- supports joins, aggregations, CTEs, views and window functions;
- is sufficient for the current 566,694-row analytical dataset;
- can be accessed directly from Python.

## Database Structure

The database will contain:

1. `complaints`
2. `product_kpis`
3. `issue_severity_kpis`
4. `company_kpis`
5. `company_product_kpis`
6. `company_product_attention`
7. `company_rankings`

Indexes and reusable SQL views will be created after loading and validating the tables.

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3
import numpy as np

PROJECT_DIR = Path(
    "/Users/ishika/Documents/AIProject"
)

NOTEBOOK8_OUTPUT_DIR = (
    PROJECT_DIR /
    "outputs" /
    "notebook8_kpis"
)

SQL_OUTPUT_DIR = (
    PROJECT_DIR /
    "outputs" /
    "sql_analytics"
)

SQL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATABASE_PATH = (
    SQL_OUTPUT_DIR /
    "financial_intelligence.db"
)

SQL_INPUT_FILES = {
    "complaints":
        NOTEBOOK8_OUTPUT_DIR /
        "complaint_intelligence_master.csv.gz",

    "product_kpis":
        NOTEBOOK8_OUTPUT_DIR /
        "product_kpis.csv",

    "issue_severity_kpis":
        NOTEBOOK8_OUTPUT_DIR /
        "issue_severity_kpis.csv",

    "company_kpis":
        NOTEBOOK8_OUTPUT_DIR /
        "company_kpis.csv",

    "company_product_kpis":
        NOTEBOOK8_OUTPUT_DIR /
        "company_product_kpis.csv",

    "company_product_attention":
        NOTEBOOK8_OUTPUT_DIR /
        "scored_company_product_attention.csv",

    "company_rankings":
        NOTEBOOK8_OUTPUT_DIR /
        "overall_company_attention_ranking.csv"
}

input_validation = pd.DataFrame({
    "table": SQL_INPUT_FILES.keys(),
    "file": [
        path.name
        for path in SQL_INPUT_FILES.values()
    ],
    "exists": [
        path.exists()
        for path in SQL_INPUT_FILES.values()
    ],
    "size_mb": [
        round(
            path.stat().st_size / (1024 ** 2),
            2
        )
        if path.exists()
        else np.nan
        for path in SQL_INPUT_FILES.values()
    ]
})

display(input_validation)

print("SQLite database will be created at:")
print(DATABASE_PATH)

if not input_validation["exists"].all():
    raise FileNotFoundError(
        "One or more Notebook 8 outputs are missing."
    )

,table,file,exists,size_mb
0,complaints,complaint_intelligence_master.csv.gz,True,18.25
1,product_kpis,product_kpis.csv,True,0.00
2,issue_severity_kpis,issue_severity_kpis.csv,True,0.01
3,company_kpis,company_kpis.csv,True,0.33
4,company_product_kpis,company_product_kpis.csv,True,0.60
5,company_product_attention,scored_company_product_attention.csv,True,0.05
6,company_rankings,overall_company_attention_ranking.csv,True,0.02


SQLite database will be created at:
/Users/ishika/Documents/AIProject/outputs/sql_analytics/financial_intelligence.db


## 3. Load Analytical Tables into SQLite

The complaint master table is loaded in chunks to control memory usage. Smaller KPI tables are loaded directly.

Existing tables with the same names are replaced, making this notebook safely rerunnable without creating duplicate rows.

In [2]:
connection = sqlite3.connect(
    DATABASE_PATH
)

# Improve bulk-loading performance
connection.execute(
    "PRAGMA journal_mode = WAL;"
)

connection.execute(
    "PRAGMA synchronous = NORMAL;"
)

connection.execute(
    "PRAGMA temp_store = MEMORY;"
)

# Load the 566,694-row complaint table in chunks
COMPLAINT_CHUNK_SIZE = 100_000

complaint_chunk_reader = pd.read_csv(
    SQL_INPUT_FILES["complaints"],
    compression="gzip",
    dtype={
        "complaint_id": "string"
    },
    chunksize=COMPLAINT_CHUNK_SIZE,
    low_memory=False
)

loaded_complaint_rows = 0

for chunk_number, chunk in enumerate(
    complaint_chunk_reader,
    start=1
):
    chunk.to_sql(
        "complaints",
        connection,
        if_exists=(
            "replace"
            if chunk_number == 1
            else "append"
        ),
        index=False,
        chunksize=5_000
    )
    
    loaded_complaint_rows += len(chunk)
    
    print(
        f"Loaded complaint chunk {chunk_number}: "
        f"{loaded_complaint_rows:,} rows"
    )

# Load the smaller analytical tables
for table_name, file_path in SQL_INPUT_FILES.items():
    if table_name == "complaints":
        continue
    
    dataframe = pd.read_csv(
        file_path,
        low_memory=False
    )
    
    dataframe.to_sql(
        table_name,
        connection,
        if_exists="replace",
        index=False,
        chunksize=5_000
    )
    
    print(
        f"Loaded {table_name}: "
        f"{len(dataframe):,} rows"
    )

connection.commit()

# Validate tables and row counts
table_names = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

row_count_results = []

for table_name in table_names["table_name"]:
    row_count = pd.read_sql_query(
        f"""
        SELECT COUNT(*) AS row_count
        FROM "{table_name}";
        """,
        connection
    ).iloc[0, 0]
    
    row_count_results.append({
        "table": table_name,
        "row_count": row_count
    })

database_validation = pd.DataFrame(
    row_count_results
)

display(database_validation)

print(
    "Database size:",
    round(
        DATABASE_PATH.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

Loaded complaint chunk 1: 100,000 rows
Loaded complaint chunk 2: 200,000 rows
Loaded complaint chunk 3: 300,000 rows
Loaded complaint chunk 4: 400,000 rows
Loaded complaint chunk 5: 500,000 rows
Loaded complaint chunk 6: 566,694 rows
Loaded product_kpis: 5 rows
Loaded issue_severity_kpis: 59 rows
Loaded company_kpis: 3,326 rows
Loaded company_product_kpis: 4,851 rows
Loaded company_product_attention: 251 rows
Loaded company_rankings: 117 rows


,table,row_count
0,company_kpis,3326
1,company_product_attention,251
2,company_product_kpis,4851
3,company_rankings,117
4,complaints,566694
5,issue_severity_kpis,59
6,product_kpis,5


Database size: 164.6 MB


## 4. Create Database Indexes

Indexes improve query performance without changing the underlying data.

Indexes are created for complaint IDs, products, companies, dates, issues, topics, sentiment labels and ranking fields. Composite indexes support common company–product and company–month queries.

In [3]:
index_statements = [
    """
    CREATE UNIQUE INDEX IF NOT EXISTS
    idx_complaints_complaint_id
    ON complaints (complaint_id);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_product
    ON complaints (product);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_company
    ON complaints (company);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_company_product
    ON complaints (company, product);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_date_received
    ON complaints (date_received);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_month
    ON complaints (complaint_month);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_issue
    ON complaints (issue);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_sentiment
    ON complaints (sentiment_label);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_topic
    ON complaints (topic_label);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_complaints_company_month
    ON complaints (company, complaint_month);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_company_product_attention_product_score
    ON company_product_attention
    (product, complaint_attention_score DESC);
    """,
    
    """
    CREATE INDEX IF NOT EXISTS
    idx_company_rankings_rank
    ON company_rankings (overall_company_rank);
    """
]

for statement in index_statements:
    connection.execute(statement)

connection.execute("ANALYZE;")
connection.commit()

index_validation = pd.read_sql_query(
    """
    SELECT
        name AS index_name,
        tbl_name AS table_name
    FROM sqlite_master
    WHERE type = 'index'
      AND name NOT LIKE 'sqlite_autoindex%'
    ORDER BY tbl_name, name;
    """,
    connection
)

display(index_validation)

print(
    "User-created indexes:",
    len(index_validation)
)

print(
    "Updated database size:",
    round(
        DATABASE_PATH.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

,index_name,table_name
0,idx_company_product_attention_product_score,company_product_attention
1,idx_company_rankings_rank,company_rankings
2,idx_complaints_company,complaints
3,idx_complaints_company_month,complaints
4,idx_complaints_company_product,complaints
5,idx_complaints_complaint_id,complaints
6,idx_complaints_date_received,complaints
7,idx_complaints_issue,complaints
8,idx_complaints_month,complaints
9,idx_complaints_product,complaints


User-created indexes: 12
Updated database size: 335.69 MB


## 5. Create Reusable Analytical Views

SQL views provide reusable, dashboard-ready query layers without duplicating the underlying data.

The following views are created:

- product KPI overview;
- high-attention company–product combinations;
- monthly product trends;
- company–topic hotspots;
- overall company ranking.

In [4]:
connection.executescript(
    """
    DROP VIEW IF EXISTS vw_product_overview;
    DROP VIEW IF EXISTS vw_high_attention_company_products;
    DROP VIEW IF EXISTS vw_monthly_product_trends;
    DROP VIEW IF EXISTS vw_company_topic_hotspots;
    DROP VIEW IF EXISTS vw_company_attention_ranking;

    CREATE VIEW vw_product_overview AS
    SELECT
        product,
        total_complaints,
        narrative_complaints,
        narrative_coverage_percentage,
        negative_sentiment_percentage,
        mean_negative_probability,
        untimely_response_percentage,
        monetary_relief_percentage,
        any_relief_percentage,
        template_heavy_percentage,
        topic_outlier_percentage
    FROM product_kpis;

    CREATE VIEW vw_high_attention_company_products AS
    SELECT
        company,
        product,
        product_rank,
        total_complaints,
        narrative_complaints,
        non_template_negative_percentage,
        mean_issue_severity_score,
        untimely_response_percentage,
        monetary_relief_percentage,
        complaint_attention_score,
        attention_tier
    FROM company_product_attention
    WHERE attention_tier IN (
        'High attention',
        'Very high attention'
    );

    CREATE VIEW vw_monthly_product_trends AS
    SELECT
        complaint_month,
        product,
        COUNT(*) AS total_complaints,
        SUM(
            CASE
                WHEN has_narrative_intelligence = 1
                THEN 1
                ELSE 0
            END
        ) AS narrative_complaints,
        SUM(
            CASE
                WHEN sentiment_label = 'negative'
                THEN 1
                ELSE 0
            END
        ) AS negative_narrative_complaints,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN sentiment_label = 'negative'
                    THEN 1
                    ELSE 0
                END
            )
            / NULLIF(
                SUM(
                    CASE
                        WHEN sentiment_label IS NOT NULL
                        THEN 1
                        ELSE 0
                    END
                ),
                0
            ),
            2
        ) AS negative_sentiment_percentage,
        SUM(
            CASE
                WHEN is_untimely = 1
                THEN 1
                ELSE 0
            END
        ) AS untimely_complaints,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN is_untimely = 1
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS untimely_response_percentage,
        SUM(
            CASE
                WHEN has_any_relief = 1
                THEN 1
                ELSE 0
            END
        ) AS relief_complaints
    FROM complaints
    GROUP BY
        complaint_month,
        product;

    CREATE VIEW vw_company_topic_hotspots AS
    SELECT
        company,
        product,
        topic_label,
        topic_type,
        COUNT(*) AS complaint_count,
        SUM(
            CASE
                WHEN sentiment_label = 'negative'
                THEN 1
                ELSE 0
            END
        ) AS negative_complaints,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN sentiment_label = 'negative'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS negative_percentage,
        ROUND(
            AVG(negative_probability),
            4
        ) AS mean_negative_probability
    FROM complaints
    WHERE topic_label IS NOT NULL
      AND topic_type != 'Not eligible'
    GROUP BY
        company,
        product,
        topic_label,
        topic_type;

    CREATE VIEW vw_company_attention_ranking AS
    SELECT
        overall_company_rank,
        company,
        total_complaints,
        narrative_complaints,
        non_template_negative_percentage,
        untimely_response_percentage,
        monetary_relief_percentage,
        scoring_coverage_percentage,
        weighted_company_attention_score,
        highest_attention_product,
        highest_product_score,
        overall_attention_tier
    FROM company_rankings;
    """
)

connection.commit()

view_validation = pd.read_sql_query(
    """
    SELECT
        name AS view_name
    FROM sqlite_master
    WHERE type = 'view'
    ORDER BY name;
    """,
    connection
)

display(view_validation)

print("\nProduct overview view")

display(
    pd.read_sql_query(
        """
        SELECT *
        FROM vw_product_overview
        ORDER BY total_complaints DESC;
        """,
        connection
    )
)

,view_name
0,vw_company_attention_ranking
1,vw_company_topic_hotspots
2,vw_high_attention_company_products
3,vw_monthly_product_trends
4,vw_product_overview



Product overview view


,product,total_complaints,narrative_complaints,narrative_coverage_percentage,negative_sentiment_percentage,mean_negative_probability,untimely_response_percentage,monetary_relief_percentage,any_relief_percentage,template_heavy_percentage,topic_outlier_percentage
0,Debt collection,283205,100610,35.53,41.59,0.4324,2.89,0.16,25.74,62.47,34.86
1,Credit card,89979,41517,46.14,54.57,0.5082,0.38,12.65,33.22,0.00,7.53
2,"Money transfer, virtual currency, or money ser...",84619,64862,76.65,81.37,0.7709,1.21,2.40,3.94,55.92,13.60
3,Checking or savings account,84194,51448,61.11,58.05,0.5512,0.72,10.58,15.41,0.00,19.75
4,Mortgage,24697,14029,56.80,52.13,0.4953,1.51,1.72,4.44,0.00,1.52


## 6. Rank Companies Within Each Product

This query uses:

- a Common Table Expression (`WITH`);
- a window function (`ROW_NUMBER`);
- product-level partitioning;
- multi-column ordering.

It returns the three highest-attention company–product combinations within each product category.

In [6]:
top_companies_by_product_query = """
WITH product_rankings AS (
    SELECT
        company,
        product,
        total_complaints,
        non_template_negative_percentage,
        mean_issue_severity_score,
        untimely_response_percentage,
        monetary_relief_percentage,
        complaint_attention_score,
        ROW_NUMBER() OVER (
            PARTITION BY product
            ORDER BY
                complaint_attention_score DESC,
                total_complaints DESC
        ) AS peer_rank
    FROM company_product_attention
)
SELECT
    product,
    peer_rank,
    company,
    total_complaints,
    non_template_negative_percentage,
    mean_issue_severity_score,
    untimely_response_percentage,
    monetary_relief_percentage,
    complaint_attention_score
FROM product_rankings
WHERE peer_rank <= 3
ORDER BY
    product,
    peer_rank;
"""

top_companies_by_product = pd.read_sql_query(
    top_companies_by_product_query,
    connection
)

display(top_companies_by_product)

,product,peer_rank,company,total_complaints,non_template_negative_percentage,mean_issue_severity_score,untimely_response_percentage,monetary_relief_percentage,complaint_attention_score
0,Checking or savings account,1,NAVY FEDERAL CREDIT UNION,10606,86.00,66.99,0.00,4.43,82.16
1,Checking or savings account,2,"Paypal Holdings, Inc",1059,55.88,66.29,0.09,14.35,73.51
2,Checking or savings account,3,"SOFI TECHNOLOGIES, INC.",1968,56.54,66.54,0.00,7.67,72.84
3,Credit card,1,SYNCHRONY FINANCIAL,6570,54.78,57.00,0.44,17.11,75.91
4,Credit card,2,GOLDMAN SACHS BANK USA,2357,59.12,61.26,0.00,27.28,74.17
5,Credit card,3,"BANK OF AMERICA, NATIONAL ASSOCIATION",4142,52.03,59.45,0.48,37.37,72.27
6,Debt collection,1,AMERICAN EXPRESS COMPANY,1206,58.82,63.34,0.33,0.50,83.92
7,Debt collection,2,ENCORE CAPITAL GROUP INC.,13486,53.81,55.89,0.01,0.02,78.28
8,Debt collection,3,"Portfolio Recovery Associates, LLC",12353,53.66,55.21,3.81,0.00,78.25
9,"Money transfer, virtual currency, or money ser...",1,"Block, Inc.",42569,67.91,54.66,1.20,0.01,88.00


## 7. Monthly Complaint Trends and Growth

This query uses multiple CTEs and the `LAG` window function to compare each month’s complaint volume with the previous month.

The four latest available months are displayed for every product. Month-over-month growth should be interpreted carefully if the final month represents incomplete data.

In [7]:
monthly_growth_query = """
WITH monthly_metrics AS (
    SELECT
        complaint_month,
        product,
        total_complaints,
        negative_sentiment_percentage,
        untimely_response_percentage,
        relief_complaints
    FROM vw_monthly_product_trends
),
growth_calculation AS (
    SELECT
        complaint_month,
        product,
        total_complaints,
        LAG(total_complaints) OVER (
            PARTITION BY product
            ORDER BY complaint_month
        ) AS previous_month_complaints,
        negative_sentiment_percentage,
        untimely_response_percentage,
        relief_complaints
    FROM monthly_metrics
),
recent_months AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY product
            ORDER BY complaint_month DESC
        ) AS recent_month_rank
    FROM growth_calculation
)
SELECT
    product,
    complaint_month,
    total_complaints,
    previous_month_complaints,
    ROUND(
        100.0
        * (
            total_complaints
            - previous_month_complaints
        )
        / NULLIF(
            previous_month_complaints,
            0
        ),
        2
    ) AS month_over_month_growth_percentage,
    negative_sentiment_percentage,
    untimely_response_percentage,
    relief_complaints
FROM recent_months
WHERE recent_month_rank <= 4
ORDER BY
    product,
    complaint_month;
"""

monthly_growth_results = pd.read_sql_query(
    monthly_growth_query,
    connection
)

display(monthly_growth_results)

,product,complaint_month,total_complaints,previous_month_complaints,month_over_month_growth_percentage,negative_sentiment_percentage,untimely_response_percentage,relief_complaints
0,Checking or savings account,2025-09,6141,6505,-5.60,53.48,0.64,1046
1,Checking or savings account,2025-10,6308,6141,2.72,52.59,0.51,1023
2,Checking or savings account,2025-11,6076,6308,-3.68,52.55,0.77,1047
3,Checking or savings account,2025-12,7692,6076,26.60,53.69,0.70,1368
4,Credit card,2025-09,8504,7919,7.39,57.39,0.22,2793
5,Credit card,2025-10,7413,8504,-12.83,55.93,0.28,2391
6,Credit card,2025-11,6861,7413,-7.45,57.01,0.60,2003
7,Credit card,2025-12,7860,6861,14.56,54.23,1.20,2124
8,Debt collection,2025-09,26816,25627,4.64,46.34,2.14,6780
9,Debt collection,2025-10,25547,26816,-4.73,40.88,2.82,5954


## 8. Connect High-Attention Companies with Their Main Complaint Themes

This query joins the company–product attention scores with topic-level complaint aggregations.

A window function identifies the largest clean discovered theme for each company–product combination. Template-heavy and outlier categories are excluded so the result highlights interpretable business themes.

In [9]:
company_topic_join_query = """
WITH high_attention AS (
    SELECT
        company,
        product,
        complaint_attention_score,
        attention_tier,
        total_complaints,
        non_template_negative_percentage
    FROM company_product_attention
    WHERE attention_tier IN (
        'High attention',
        'Very high attention'
    )
),
ranked_topics AS (
    SELECT
        company,
        product,
        topic_label,
        complaint_count,
        negative_complaints,
        negative_percentage,
        mean_negative_probability,
        ROW_NUMBER() OVER (
            PARTITION BY company, product
            ORDER BY
                complaint_count DESC,
                negative_complaints DESC
        ) AS topic_rank
    FROM vw_company_topic_hotspots
    WHERE topic_type = 'Discovered theme'
  AND complaint_count >= 100
)
SELECT
    h.company,
    h.product,
    h.total_complaints,
    h.complaint_attention_score,
    h.attention_tier,
    h.non_template_negative_percentage,
    t.topic_label AS largest_discovered_topic,
    t.complaint_count AS topic_complaints,
    t.negative_complaints AS topic_negative_complaints,
    t.negative_percentage AS topic_negative_percentage
FROM high_attention AS h
LEFT JOIN ranked_topics AS t
    ON h.company = t.company
   AND h.product = t.product
   AND t.topic_rank = 1
ORDER BY
    h.complaint_attention_score DESC
LIMIT 20;
"""

company_topic_results = pd.read_sql_query(
    company_topic_join_query,
    connection
)

display(company_topic_results)

,company,product,total_complaints,complaint_attention_score,attention_tier,non_template_negative_percentage,largest_discovered_topic,topic_complaints,topic_negative_complaints,topic_negative_percentage
0,"Block, Inc.","Money transfer, virtual currency, or money ser...",42569,88.00,Very high attention,67.91,Cash App Account Takeovers and Unauthorized Tr...,5490.0,2913.0,53.06
1,"Paypal Holdings, Inc","Money transfer, virtual currency, or money ser...",5482,86.00,Very high attention,60.14,"PayPal Refunds, Seller Disputes and Frozen Funds",2226.0,1383.0,62.13
2,AMERICAN EXPRESS COMPANY,Debt collection,1206,83.92,Very high attention,58.82,None,NaN,NaN,NaN
3,NAVY FEDERAL CREDIT UNION,Checking or savings account,10606,82.16,Very high attention,86.00,Navy Federal Transaction Posting and Fee Pract...,5049.0,5025.0,99.52
4,ENCORE CAPITAL GROUP INC.,Debt collection,13486,78.28,Very high attention,53.81,None,NaN,NaN,NaN
5,"Portfolio Recovery Associates, LLC",Debt collection,12353,78.25,Very high attention,53.66,None,NaN,NaN,NaN
6,"CCS Financial Services, Inc.",Debt collection,6450,78.06,Very high attention,55.13,"Medical Debt, Insurance Billing and Collections",151.0,66.0,43.71
7,JPMORGAN CHASE & CO.,Debt collection,1826,76.59,Very high attention,52.94,None,NaN,NaN,NaN
8,SYNCHRONY FINANCIAL,Credit card,6570,75.91,Very high attention,54.78,"General Card Billing, Payment and Credit-Repor...",3828.0,2091.0,54.62
9,Freedom Mortgage Company,Mortgage,1180,75.74,Very high attention,53.39,"General Mortgage Servicing, Payment and Escrow...",690.0,374.0,54.20


## 9. Export SQL Database, Schema and Query Results

The completed SQLite database, database-definition script, portfolio SQL queries and query-result tables are exported as reusable deliverables.

The `.db` file contains the actual relational database. The `.sql` files make the schema, indexes, views and analytical queries visible for portfolio review.

In [10]:
SQL_QUERY_RESULTS_DIR = (
    SQL_OUTPUT_DIR /
    "query_results"
)

SQL_QUERY_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SCHEMA_PATH = (
    SQL_OUTPUT_DIR /
    "financial_intelligence_schema.sql"
)

QUERIES_PATH = (
    SQL_OUTPUT_DIR /
    "financial_intelligence_queries.sql"
)

TOP_COMPANIES_RESULT_PATH = (
    SQL_QUERY_RESULTS_DIR /
    "top_companies_by_product.csv"
)

MONTHLY_GROWTH_RESULT_PATH = (
    SQL_QUERY_RESULTS_DIR /
    "monthly_product_growth.csv"
)

COMPANY_TOPIC_RESULT_PATH = (
    SQL_QUERY_RESULTS_DIR /
    "high_attention_company_topics.csv"
)

# Extract table, index and view definitions
schema_objects = pd.read_sql_query(
    """
    SELECT
        type,
        name,
        sql
    FROM sqlite_master
    WHERE sql IS NOT NULL
      AND name NOT LIKE 'sqlite_%'
    ORDER BY
        CASE type
            WHEN 'table' THEN 1
            WHEN 'index' THEN 2
            WHEN 'view' THEN 3
            ELSE 4
        END,
        name;
    """,
    connection
)

schema_sql = (
    "-- AI Financial Intelligence Analyst\n"
    "-- SQLite schema, indexes and analytical views\n\n"
    + "\n\n".join(
        statement.rstrip(";") + ";"
        for statement in schema_objects["sql"]
    )
)

SCHEMA_PATH.write_text(
    schema_sql,
    encoding="utf-8"
)

# Export the portfolio analytical queries
queries_sql = f"""
-- AI Financial Intelligence Analyst
-- Portfolio SQL analytical queries

-- Query 1: Top three companies within each product
{top_companies_by_product_query.strip()}

-- Query 2: Monthly product growth using LAG
{monthly_growth_query.strip()}

-- Query 3: High-attention companies joined to
-- sufficiently large discovered topic hotspots
{company_topic_join_query.strip()}
""".strip()

QUERIES_PATH.write_text(
    queries_sql,
    encoding="utf-8"
)

# Export query results
top_companies_by_product.to_csv(
    TOP_COMPANIES_RESULT_PATH,
    index=False
)

monthly_growth_results.to_csv(
    MONTHLY_GROWTH_RESULT_PATH,
    index=False
)

company_topic_results.to_csv(
    COMPANY_TOPIC_RESULT_PATH,
    index=False
)

# Consolidate the WAL file into the main database
connection.commit()

connection.execute(
    "PRAGMA wal_checkpoint(TRUNCATE);"
)

connection.commit()

sql_deliverables = [
    DATABASE_PATH,
    SCHEMA_PATH,
    QUERIES_PATH,
    TOP_COMPANIES_RESULT_PATH,
    MONTHLY_GROWTH_RESULT_PATH,
    COMPANY_TOPIC_RESULT_PATH
]

sql_export_summary = pd.DataFrame({
    "file": [
        path.name
        for path in sql_deliverables
    ],
    "exists": [
        path.exists()
        for path in sql_deliverables
    ],
    "size_mb": [
        round(
            path.stat().st_size / (1024 ** 2),
            2
        )
        if path.exists()
        else np.nan
        for path in sql_deliverables
    ]
})

display(sql_export_summary)

print("SQL output directory:")
print(SQL_OUTPUT_DIR)

,file,exists,size_mb
0,financial_intelligence.db,True,335.71
1,financial_intelligence_schema.sql,True,0.01
2,financial_intelligence_queries.sql,True,0.00
3,top_companies_by_product.csv,True,0.00
4,monthly_product_growth.csv,True,0.00
5,high_attention_company_topics.csv,True,0.00


SQL output directory:
/Users/ishika/Documents/AIProject/outputs/sql_analytics


In [11]:
connection.close()

print("SQLite connection closed successfully.")
print("Database saved at:")
print(DATABASE_PATH)

SQLite connection closed successfully.
Database saved at:
/Users/ishika/Documents/AIProject/outputs/sql_analytics/financial_intelligence.db


## 10. Conclusion

This notebook created a complete SQLite analytics layer for the AI Financial Intelligence Analyst project.

### Database Deliverables

The final database contains:

- 566,694 complaint-level records;
- seven relational analytical tables;
- twelve performance indexes;
- five reusable SQL views;
- product, issue, company and company–product KPI tables;
- topic and sentiment intelligence;
- composite complaint-attention rankings.

### SQL Capabilities Demonstrated

The analytical queries demonstrated:

- filtering and aggregation;
- joins across analytical views;
- Common Table Expressions;
- window functions;
- product-level partitioning;
- company ranking;
- `LAG`-based monthly growth;
- conditional aggregation;
- null-safe division;
- topic-level complaint analysis.

### Analytical Views

Reusable views were created for:

- product KPI overviews;
- high-attention company–product combinations;
- monthly product trends;
- company–topic hotspots;
- overall company attention rankings.

### Interpretation

The SQL layer makes the project outputs queryable without rerunning sentiment analysis, embedding generation or topic modelling.

Attention scores remain exploratory prioritisation indicators. They do not establish company misconduct, regulatory violations, financial instability or customer-level incidence rates.

### Final Output

The portable `financial_intelligence.db` database can now support:

- dashboard development;
- Streamlit applications;
- scheduled analytical reporting;
- portfolio SQL demonstrations;
- future RAG or natural-language-to-SQL functionality.